# FRAUD DETECTION 

A classic problem in financial environments. We will take a look at classifying transitions into fraud and not fraud using scikit-learn package.

This notebook will be using a synthetic database found on kaggle for educational purposes because we all need practice :-).   
The data can be found here: https://www.kaggle.com/datasets/jayjoshi37/digital-payment-fraud-detection/data

I will break this proces down into the follwoing steps:

1. Data Exploration 
2. Data prepartion and pre-processing (including feature engineering)
3. Modelling 
4. Evaluation and testing 

We have the following fields within our dataset: 

transaction_id - identifier to each transaction.   
user_id - identifier for each user.   
transaction_amount - amount for each transaction.   
transaction_type - how funds were exchanged e.g "payment" or "bank transfer".   
payment_mode - wallet, card, UPI etc.   
device_type - device transaction was made from e.g iOS.   
device_location - location of the device used to make transaction.   
account_age_days - age of the account.     
transaction_hour - time of transaction in 24 hour notation.    
previous_failed_attempts - if there were previous attempts to make fraudulent transactions.   
avg_transaction_amount - avg amount each account usually makes.   
is_international - is the trasnaction international.    
ip_risk_score - a numerical value  that quantifies the likelihood an IP address is involved in malicious activity, such as fraud, spam, or cyberattacks. 
login_attempts_last_24h - number of login attempts to the account in the last 24 hours.   
fraud_label- is the transaction fraud or not.


## Data Exploration 

In [1]:
# First we need to start by acquiring all of our dependecies 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Now we will load our data into our notebook for modelling:
current_dir = Path.cwd()
print(current_dir)              # this directory is 1 level too low in the project structure
parent_dir = current_dir.parent # go up 1 level in the structure 
print(parent_dir)               # now we are in the parent directory that we can now use to naviagte into the dataset 
file_path = parent_dir/ "dataset" / "Digital_Payment_Fraud_Detection_Dataset.csv" 
print(file_path)                # now we are in the correct bracnh of the tree where our dataset is

fraud = pd.read_csv(file_path)  # IT WORKED!!!


/Users/leta/Desktop/Data Science Career/Python/Python Projects/FraudDetection/modelling
/Users/leta/Desktop/Data Science Career/Python/Python Projects/FraudDetection
/Users/leta/Desktop/Data Science Career/Python/Python Projects/FraudDetection/dataset/Digital_Payment_Fraud_Detection_Dataset.csv


In [ ]:
fraud.shape # dimensions of the dataframe

(7500, 15)

In [ ]:
# Let us take a look into the data we have 
fraud.head()

,transaction_id,user_id,transaction_amount,transaction_type,payment_mode,device_type,device_location,account_age_days,transaction_hour,previous_failed_attempts,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,fraud_label
0,T1,U3756,18758.28,Transfer,UPI,Web,Hyderabad,895,14,1,25535.84,0,0.718,4,0
1,T2,U7899,47538.18,Payment,Wallet,iOS,Hyderabad,918,21,0,3955.85,0,0.525,9,0
2,T3,U1765,36613.10,Payment,Card,Android,Chennai,1506,8,4,22727.71,0,0.985,9,0
3,T4,U8850,29952.99,Payment,Wallet,iOS,Chennai,800,1,3,18095.89,0,0.797,2,0
4,T5,U9049,7843.13,Payment,UPI,Web,Delhi,301,4,1,9317.49,1,0.468,1,0


In [ ]:
# In this dataset we have mostly numerical variables with a few catgeroical variables.
# Let us look at what fields we have
fraud.keys() #column names of us data


Index(['transaction_id', 'user_id', 'transaction_amount', 'transaction_type',
       'payment_mode', 'device_type', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts',
       'avg_transaction_amount', 'is_international', 'ip_risk_score',
       'login_attempts_last_24h', 'fraud_label'],
      dtype='str')

In [6]:
#What unique labels do we have in the dataset 
fraud.nunique()
# The fields with the highest unique values are the ID fields, transaction amounts, account ages, avgerage transaction amounts, ip risk scores 
# ID fields are just unqiue ientifiers but have no bearing on prediction therefore we will drop these fields especially since they have many unique values.  
# We will plore some more of these fields to see there bearing on the predcitve power of our models using exploratory data analysis and data mining.

transaction_id              7500
user_id                     5106
transaction_amount          7499
transaction_type               3
payment_mode                   4
device_type                    3
device_location                5
account_age_days            1943
transaction_hour              24
previous_failed_attempts       5
avg_transaction_amount      7498
is_international               2
ip_risk_score               1000
login_attempts_last_24h        9
fraud_label                    2
dtype: int64

In [ ]:
# Finally let us see of the data is balanced or not 
fraud["fraud_label"].value_counts(normalize=True) #count all fraud labels and get proportion

# Fraud - 6.52% of observations    
# Non fraud - 93.28% of observations 

# Our data is very skewed towards non fraud transactions.
# This will infrom our metric to measure perfomrance (probably balanced accuarcy, recall and precision) 
# as well as how we stratify the data when partitioning into test and train sets. 


fraud_label
0    0.9348
1    0.0652
Name: proportion, dtype: float64

Our variables with categorical information being: transaction_type, payment_mode, device_type and device_location do not have too many unqiue values.  
This infroms what methods we can use to deal with these values for classification models that only use numerical data.  
- A possible solution would be to use dummy variables. 

##  DATA PREPARATION AND PRE-PROCESSING 


FEATURE ENGINEERING   
We will now create features that improve the performance of our models based on the EDA we conducted in R and the insights we gathered. 

In [ ]:
# Now we will take partition the data into dependent and indepenedt variables. 
x =fraud[['user_id', 'transaction_amount', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts', 'payment_mode', 'device_type', 'transaction_type',
       'avg_transaction_amount', 'is_international', 'ip_risk_score',
       'login_attempts_last_24h']]
y = fraud[['fraud_label']]                               # subsetting our predicted variable 

In [78]:
# Partitioning our data 
# The data will be partitioned into tran and test splits at a 70/30% proportion 

from sklearn.model_selection import train_test_split # package to spilt our data 

x_train, x_test, y_train, y_test = train_test_split(x,y,              # our dependent and independent variables to be used
                                                    stratify = y,     # keeping proportion of fraud and not fraud equal in the the train and test sets
                                                    random_state=42,  # setting a seed for reproducibility 
                                                    test_size=0.3)    # 30% test size


In [79]:
x_train.keys() 
# Making sure we have all of the columns necessary in our trainng set with no data leakage.

Index(['user_id', 'transaction_amount', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts', 'payment_mode',
       'device_type', 'transaction_type', 'avg_transaction_amount',
       'is_international', 'ip_risk_score', 'login_attempts_last_24h'],
      dtype='str')

In [80]:
y_train.keys()
# Our labelled dependent variable 

Index(['fraud_label'], dtype='str')

In [ ]:
#get unique ids
#groups x_train rows by those ids
#sum each column 
#make that a new dataframe with proportions of each device location (maybe have a small value for )
ids = x_train['user_id'].unique().tolist()
dataframe1 = pd.DataFrame()
 
dataframe1 = x_train.sort_values(by = "user_id")
dataframe1 = pd.get_dummies(data = dataframe1, columns= [ "device_location"], dtype = int)


['U5261',
 'U8927',
 'U1965',
 'U4533',
 'U6077',
 'U2316',
 'U7935',
 'U3025',
 'U8804',
 'U1063',
 'U4882',
 'U1386',
 'U8416',
 'U4469',
 'U5499',
 'U4197',
 'U4822',
 'U8712',
 'U1015',
 'U6838',
 'U6263',
 'U1499',
 'U5569',
 'U7682',
 'U8752',
 'U7507',
 'U6917',
 'U6672',
 'U4366',
 'U1730',
 'U6197',
 'U5789',
 'U3075',
 'U4577',
 'U5941',
 'U7033',
 'U3340',
 'U5997',
 'U7650',
 'U1619',
 'U5815',
 'U5890',
 'U4615',
 'U7571',
 'U5778',
 'U9688',
 'U1337',
 'U1699',
 'U5204',
 'U5401',
 'U1264',
 'U5632',
 'U5201',
 'U3012',
 'U9380',
 'U9193',
 'U6158',
 'U6039',
 'U6552',
 'U6546',
 'U6188',
 'U1943',
 'U9855',
 'U5702',
 'U9389',
 'U9490',
 'U8358',
 'U2132',
 'U3890',
 'U9492',
 'U3234',
 'U1407',
 'U9460',
 'U7276',
 'U4020',
 'U7685',
 'U1801',
 'U3222',
 'U3227',
 'U2989',
 'U7885',
 'U6136',
 'U9243',
 'U1141',
 'U6195',
 'U9063',
 'U9944',
 'U5163',
 'U7743',
 'U2963',
 'U4114',
 'U4268',
 'U9575',
 'U2709',
 'U7385',
 'U3937',
 'U7839',
 'U2553',
 'U6871',
 'U2794',


In [167]:
dataframe1

,user_id,transaction_amount,account_age_days,transaction_hour,previous_failed_attempts,payment_mode,device_type,transaction_type,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,device_location_Bangalore,device_location_Chennai,device_location_Delhi,device_location_Hyderabad,device_location_Mumbai
3866,U1000,20687.97,864,4,4,NetBanking,Web,Withdrawal,3497.06,0,0.127,7,1,0,0,0,0
4753,U1007,25469.26,720,4,2,NetBanking,Android,Payment,26044.93,0,0.893,2,0,1,0,0,0
1453,U1008,28551.96,710,11,3,Card,Android,Withdrawal,17232.27,1,0.082,1,1,0,0,0,0
4831,U1008,36986.49,769,3,2,Wallet,Web,Payment,27085.88,0,0.743,2,0,0,0,0,1
2452,U1009,27907.56,422,3,1,Wallet,Web,Payment,23113.49,0,0.286,5,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5306,U9997,23875.32,447,10,0,UPI,Android,Withdrawal,25250.44,0,0.402,2,0,0,0,0,1
1250,U9997,39837.04,1064,19,0,UPI,Android,Payment,1038.08,0,0.234,5,0,1,0,0,0
5953,U9997,30833.72,769,13,3,Card,Web,Transfer,17255.61,0,0.358,9,0,0,0,1,0
3721,U9998,41736.86,1155,16,4,UPI,Web,Payment,16845.71,0,0.661,9,0,0,1,0,0


In [168]:
dataframe1 = dataframe1.set_index("user_id")

In [169]:
dataframe1.index

Index(['U1000', 'U1007', 'U1008', 'U1008', 'U1009', 'U1012', 'U1013', 'U1013',
       'U1015', 'U1017',
       ...
       'U9987', 'U9988', 'U9991', 'U9992', 'U9995', 'U9997', 'U9997', 'U9997',
       'U9998', 'U9998'],
      dtype='str', name='user_id', length=5250)

In [ ]:
user1015 = dataframe1.loc['U1015',  'device_location_Bangalore':'device_location_Mumbai'].to_list()
dataframe2 = pd.DataFrame(ids, columns = ["user_id"])



list

### PIPELINES
We will now experiment with making a scikit-learn pipeline.  
These are methods we can use to automate the data scince process by doing things like ETL, Pre-processing, Feature engineering and much more. 

We will create our own class to carry out transfromations on the train and test data separetely and then fit our model.   
This will inclove the use of:   
 - FunctionTransformer
 - One hot encoding (replcaing dummy variables in a previous iteration)
 - Standrad scaler so values like transaction amounts don't skew our model 

Let's get into it!!

In [13]:
# We mist start by imprting our pipeline functionality 
from sklearn.pipeline import Pipeline                   # creating a data pre-propcessing pipeline for our test set 
from sklearn.preprocessing import FunctionTransformer   # to create feature engineering functions 

In [17]:
#Defining the class

class ETL(object): 
    def __init__(self, x):
        self.x = x

    def location_tag(x):
        #creating the location buckets 
        fraud_location = {  
        "Mumbai":5,     # highest likelihood of fraud 
        "Hyderabad" :4, # second highest likelihood of fraud etc
        "Chennai" : 3,
        "Bangalore" : 2,
        "Delhi" : 1}
        x["fraud_bucket_location"] = x["device_location"].map(fraud_location)
        return x #returning our processed test dataframe 
 
    def risk_buckets(x):
        #creating our risk buckets 
        x["ip_risk_bucket"] = np.where((x['ip_risk_score'] < 0.51) | 
                                        (x['ip_risk_score'] > 0.86), 1, 0)        # assign a 1 if the risk is in the dangerous category and 0 itherwise 

        #creating the account age buckets 
        x["account_age_bucket"] = np.where((x['account_age_days'] >= 1190 ) &
                                            (x['account_age_days'] <= 1855), 1,0) # assign a 1 if we have an account in the most denagrous range and 0 otherwise
        return x
    
    def amount_averages(x):
        transaction_average = pd.DataFrame()                        # empty dataframe to store averages 

        transaction_average["user_id"] = x['user_id'].unique()      # this represents each users average transaction amount for this round of transactions 

        means = x.groupby(['user_id'])['transaction_amount'].mean() # get the mean of each transaction amount from the test data for each User ID

        transaction_average["days_average"] = transaction_average["user_id"].map(means) # add this mean called "days_average" to the dataframe 

        averages_per_user = x.groupby(['user_id'])["avg_transaction_amount"].unique()   # conduct the same procedure for the historic known average of User IDs

        typical_average = averages_per_user.apply(np.mean)                              # get the mean of each average since some User IDs have multiple historic averages 

        transaction_average["typical_average"] = transaction_average["user_id"].map(typical_average) # add this updated historic average to the dataframe 
        x = x.merge(transaction_average, on = "user_id", how = 'left')                               # add this dataframe to x using joins and duplicate some for each User ID so left join
        return x
    
    def transaction_outliers(x):
        days_sd = np.std(x['days_average']) # standard deviation for the days average column (transaction amounts follow roughly uniform distribution form the EDA)

        outlier_days_avg = pd.DataFrame()   # empty dataframe to get the outlier transactions 

        outlier_days_avg["outlier_days_average"] = np.where( (x['transaction_amount'] <= x['days_average'] -  2.5* days_sd) | 
            (x['transaction_amount'] >= x['days_average'] + 2.5 * days_sd), 1,0)           #outliers on days average are transaction amounts outside 2.5 standard deviations from the mean
        
        typical_sd = np.std(x['typical_average']) # similar to the above 

        outlier_typical_avg = pd.DataFrame()
        outlier_typical_avg["outlier_typical_average"] = np.where( (x['transaction_amount'] <= x['days_average'] -  2.5* typical_sd) | 
            (x['transaction_amount'] >= x['days_average'] + 2.5 * typical_sd), 1,0)         # outliers outside 2 standard deviations from the mean are flagged with 1 otherwise 0
        
        x = pd.concat([x,outlier_days_avg,outlier_typical_avg ], axis =1)                   # adding the outliers in terms of the aveage amounts to the test dataframe x
        return x
    
    def international_outliers(x):
        commom_international = x.groupby('user_id')['is_international'].agg(lambda x: pd.Series.mode(x)[0]) # get the most common location (domestric or international) where each User does transactions from
        x['transaction_usual_international'] = x["user_id"].map(commom_international)                       # add this mode to the test dataframe
        x["unexpected_location"] = np.where((x['is_international'] == x['transaction_usual_international'] ) , 0,1) # assign 0 if the transaction is in the usual location and 1 otherwise
        return x

    def login_outliers(x):
        usual_login_attempts = x.groupby('user_id')['previous_failed_attempts'].mean()          # mean of the number of number of attempts to login by User ID 
        x['usual_login_attempts'] = x["user_id"].map(usual_login_attempts)                      # map these onto the test dataframe 
        x["unexpected_login_attempts"] = np.where((x['previous_failed_attempts'] <= x['usual_login_attempts'] ) , 0,1) #assign 0 is the transactions attempts are less than or equal to the normla amount and 1 otherwise 
        return x
    
    def drop_features(x):                           # drop features that we used to create our features like "user id"
        print(type(x))
        x= x.drop(['user_id','payment_mode', 'device_type', 'transaction_type', 'transaction_usual_international','usual_login_attempts' ], axis = 1)
        return x

In [ ]:
#scaling and get dummies (one hot encoding our variables)
numeric_features = ['transaction_amount', 'account_age_days',
                     'transaction_hour', 'ip_risk_score', 'days_average','typical_average'] 
    
categorical_features = ['device_location', 'previous_failed_attempts',  'is_international', 'login_attempts_last_24h']

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformation = StandardScaler()
categorical_transformation = OneHotEncoder()

preprocess_data = ColumnTransformer(
    transformers= ([
        ("num" , numeric_transformation, numeric_features),
        ("cat", categorical_transformation,categorical_features)
    ]))

In [ ]:
locationtransformer = FunctionTransformer(ETL.location_tag)
riskransformer = FunctionTransformer(ETL.risk_buckets)
amount_avgtransformer = FunctionTransformer(ETL.amount_averages)
transaction_outliertransformer = FunctionTransformer(ETL.transaction_outliers)
international_outliertransformer = FunctionTransformer(ETL.international_outliers)
login_outliertransformer = FunctionTransformer(ETL.login_outliers)

droptransformer = FunctionTransformer(ETL.drop_features)

pipeline = Pipeline([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer,),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
    ("Logistic_regression", LogisticRegression())
])

## MODELLING 

In [ ]:
#We will now us the pipleine on our test data 
pipeline.fit(x_train, y_train)

## EVALUATION AND TESTING 
We have trained our model on the train set.   
We can now use the metrics mentined earlier like balanced accuracy, precision and recall to judge our models

**Precision** - of all the positievs identified how many were truly positive?  
**Recall**- of all possible positive instances, how many did the model catch? 

For fraud detection, we value recall as our main metric.   
This is beacuse catching every fraudlent trasaction allows for the least damage to our client base as opposed to flagging a genuine transaction as fraud that can be reversed wuth no harm to the user. 

In [ ]:
y_pred_log = pipeline.predict(x_test) #making predictions on the train data 


In [ ]:
# Importing balanced accuracy
from sklearn.metrics import accuracy_score,balanced_accuracy_score, precision_recall_curve, confusion_matrix,precision_score, recall_score, ConfusionMatrixDisplay #importing all necesarry metrics

In [ ]:
accuracy_score(y_test, y_pred_log) #we have a very high accuracy score of 0.935


In [ ]:
balanced_accuracy_score(y_test,y_pred_log) #our balaned accuracy score is very lowe at 0.5

In [ ]:
precision_score(y_test,y_pred_log) # precision is 0%

In [ ]:
recall_score(y_test,y_pred_log) #recall is 0%

This indicates that the imbalance in the dataset even after stratifying the data.     
This indicates that our model learned to predict non fraud every time (class 0) to get a high accuracy bot lowe balanced accuracy, precision and recall   
Therefore we need to find another way to ensure we can identify all the positive cases. 
# We can award a greater penalty to predicting a very imbalanced outcome 

In [ ]:
pipeline2 = Pipeline([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer,),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
    ("Logistic_regression", LogisticRegression(class_weight= 'balanced'))
])



In [ ]:
pipeline2.fit(x_train, y_train)

In [ ]:
y_pred2_log = (pipeline2.predict_proba(x_test)[:,1] >= 0.35).astype(int)

In [ ]:
balanced_accuracy_score(y_test, y_pred2_log) #Our model still does not have the best balanced accuracy at 52.0%

In [ ]:
precision_score(y_test, y_pred2_log) # precision is also very low 7.03%

In [ ]:
recall_score(y_test, y_pred2_log) #however, our recall is much higher therefore, we have caught more our our fraud cases 51.7%

In [ ]:
confusion_matrix_log = ConfusionMatrixDisplay.from_predictions( #creating confusion matrix diagram from predictions
    y_test,                                                     #comparing y test data to our predcited values from the logistic regression model
    y_pred2_log
)

plt.title("Confusion matrix on from Logistic Regression")  #setting up the lablels 
plt.show()

In conculsion - even after we add extra penalty to predict each class as good as possible. 
We will now try to use another classification model 

In [ ]:
# Now we will try to use a Random Forest Model 
from sklearn.ensemble import GradientBoostingClassifier #using a Random Forest model

In [ ]:
featuretransformer3 = FunctionTransformer(new_features)

pipeline3 = Pipeline([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer,),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
    ("Gradient_boost", GradientBoostingClassifier(n_estimators= 500,
                                                       loss="exponential"))
])  #having the penalty between classes being equal


In [ ]:
pipeline3.fit(x_train, y_train)

In [ ]:
y_pred_gradient_boost = pipeline3.predict(x_test)

In [ ]:
balanced_accuracy_score(y_test, y_pred_gradient_boost) #is lower than our logistic regression with classes weighed equally 

In [ ]:
precision_score(y_test, y_pred_gradient_boost) #also very low 0

In [ ]:
recall_score(y_test, y_pred_gradient_boost) #low like our logistic regression at 0

In [ ]:
confusion_matrix_rando = ConfusionMatrixDisplay.from_predictions(
    y_test,                #comparing y test data to our predcited values from the random forest model
    y_pred_gradient_boost
)
plt.title("Confusiom Matrix of Gradient boosting")
plt.show()

Both our models minimise error by just predicting not fraud for all the transaction since the data is so skewed in not fraud approximatley 94/6 %.
Therefore since we have such unbalanced data we have to give a much larger penatly to make our models predict both classes more accuractely.   

## MODEL WITH GREATER PENALTY 

In [ ]:
# We will now create both models with a greater penatly for predicting wrong in our fraud class as well as lower our acceptance threshold for fraud.
featuretransformer4 = FunctionTransformer(new_features)

pipeline4 = Pipeline([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer,),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
    ("Logistic_regression", LogisticRegression(class_weight= {0:1, 1:10}))
])

featuretransformer5 = FunctionTransformer(new_features)

pipeline5 = Pipeline([
    ("feature_engineering", featuretransformer5),
    ("preprocess", preprocess_data),
    ("Radom_forest", RandomForestClassifier(class_weight= {0:1, 1:10}))
])

In [ ]:
algos = [pipeline4, pipeline5] #list of our pipelines on training data 
pipeline_names = ["Logistic regression pipeline" , "Random Forest pipeline"]
b_accuracy =[]        #empty list of balanced accuracy scores 
recall_metric = []    #empty list of recall  scores  
precision_metric = [] #empty list of precision  scores
threshold = 0.05

for a, p  in zip(algos, pipeline_names):  #loop that fits both models and appends balanced accuracy, precsion and recall to the empty lists 
    a.fit(x_train, y_train)
    predictions = (a.predict_proba(x_test)[:,1] >= threshold).astype(int)
    acc = balanced_accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions)
    rec = recall_score(y_test, predictions)
    
    b_accuracy.append(acc)
    recall_metric.append(rec)
    precision_metric.append(prec)

    
    confusion_matrix = ConfusionMatrixDisplay.from_predictions(
                y_test,                #comparing y test data to our predcited values from the random forest model
                predictions
            )
    plt.title(f"Confusion matrix of {p}")
    plt.show()


In [ ]:
main_metrics =pd.DataFrame()                                                #create an empty dataframe to add metrics to 
main_metrics["Description"] = [ "Balanced accuracy", "Recall", "Precision"] #create columns of metrc names  
main_metrics["Logistic Regression"] = [b_accuracy[0], recall_metric[0], precision_metric[0]] #add metrics for Logistic regression model
main_metrics["Random Forest"] = [b_accuracy[1], recall_metric[1], precision_metric[1]]       #add metrics for Random forest 


main_metrics

Okay, there are many takeaways from this experiment:   

1) Our Logistic regression model is very aggressive. It is able to catch all of the fraud cases at 100% recall (which is great) but it does so by guessing everything as fraud. 
- This means that every custimer would have every transactions falgged as fraud if this model was implemented which is a horrendous user expirience. 
- As a result the balanced accuracy falls since we got 100% of 50% and 0% of 50% of predcitions correct. 

2) Our Random Forest is much better with a recall at 70.7.0% so it catches the majority of fraud. 
-  However we still have many majority of non fraud cases being calssified as fraud so it will still be very inconveneint for all of our users.   

**All 3 models perform relatively poorly in atleast 2 metrics that are the most important.**. 

Therefore we can use resampling techniques to solve this issue.   
For severly unbalanced data, there is a resampling technique called SMOTE resampling. 

### SMOTE 
Synthetic Minority Over-Sampling Technique (SMOTE)

SMOTE is a data-level resampling technique that generates synthetic (artificial) samples for the minority class.   
Instead of simply duplicating existing examples, it creates new data points by interpolating between existing ones.   
This approach allows the model to learn broader patterns and reduces the risk of overfitting to repeated samples

In [ ]:
#First let us start by importing the necessary library 
from imblearn.over_sampling import SMOTE 
from imblearn.pipeline import Pipeline as Pipe

In [ ]:
# Now we will resample our data 
# This will involve adding the resampling step midway through into our pipeline 

featuretransformer_SMOTE = FunctionTransformer(new_features)


smt = SMOTE(sampling_strategy="minority", random_state= 40)

pipeline6 = Pipe([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer,),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
     ("smote", smt),
    ("Logistic_regression", LogisticRegression(class_weight= {0:1, 1:1.20}))
])

pipeline7 = Pipe([
    ("feature_engineering", featuretransformer_SMOTE),
    ("preprocess", preprocess_data),
    ("smote", smt),
    ("Random Forest", RandomForestClassifier(class_weight= "balanced"))
])

pipeline8 = Pipe([
    ("feature_engineering", featuretransformer_SMOTE),
    ("preprocess", preprocess_data),
    ("smote", smt),
    ("Gardient Boosting", GradientBoostingClassifier(n_estimators= 500,
                                                     max_depth= 3,
                                                     min_samples_split= 2,
                                                     learning_rate= 0.03))
])

In [ ]:
pipeline6.fit(x_train,y_train)

In [ ]:
y_pred_2 = (pipeline6.predict_proba(x_test)[:,1] >= 0.47).astype(int)

In [ ]:
balanced_accuracy_score(y_test, y_pred_2) #is lower than our logistic regression with classes weighed equally 

In [ ]:
precision_score(y_test, y_pred_2) #also very low 0

In [ ]:
recall_score(y_test, y_pred_2) #low like our logistic regression at 0

In [ ]:
confusion_matrix = ConfusionMatrixDisplay.from_predictions(
                y_test,                #comparing y test data to our predcited values from the random forest model
                y_pred_2
            )
plt.title(f"Confusion matrix of {p}")
plt.show()

In [ ]:
counts = y_train_s.value_counts() #getting the count of our resampled fraud and non fraud data  points

plt.bar(range(len(counts)),counts.values, color = ['blue','red']) #plotting our resampled data on a bar graph 
plt.xticks([0,1],['Non-fraud','Fraud'])
plt.show()

### Now we will re-run our models on the resampled data

In [ ]:
algos = [pipeline6, pipeline7, pipeline8] #list of our pipelines on training data 
pipeline_names = ["Logistic regression SMOTE pipeline" , "Random Forest SMOTE pipeline", "Gradient boosting SMOTE pipeline"]
smt_accuracy =[]        #empty list of balanced accuracy scores 
smt_recall = []    #empty list of recall  scores  
smt_precision = [] #empty list of precision  scores
threshold = 0.05

for a, p  in zip(algos, pipeline_names):  #loop that fits both models and appends balanced accuracy, precsion and recall to the empty lists 
    a.fit(x_train, y_train)
    predictions = (a.predict_proba(x_test)[:,1] >= threshold).astype(int)
    acc = balanced_accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions)
    rec = recall_score(y_test, predictions)
    
    smt_accuracy.append(acc)
    smt_recall.append(rec)
    smt_precision.append(prec)

    
    confusion_matrix = ConfusionMatrixDisplay.from_predictions(
                y_test,                #comparing y test data to our predcited values from the random forest model
                predictions
            )
    plt.title(f"Confusion matrix of {p}")
    plt.show()

In [ ]:
main_metrics_resample = pd.DataFrame()                                                            #create an empty dataframe to add metrics to 
main_metrics_resample["Description"] = [ "Balanced accuracy", "Recall", "Precision"]              #create columns of metrc names  
main_metrics_resample["Logistic Regression"] = [smt_accuracy[0], smt_recall[0], smt_precision[0]] #add metrics for Logistic regression model
main_metrics_resample["Random Forest"] = [smt_accuracy[1], smt_recall[1], smt_precision[1]]       #add metrics for Random forest 
main_metrics_resample["Gradient Boosting"] = [smt_accuracy[2], smt_recall[2], smt_precision[2]] 

main_metrics_resample

After resampling our Random Forest has the best balance of recall (approximately 76 - 79%) with the lowest number of false positive.    
Resampling did not work as well as I thought it would. 

Since our data is so unblanaved before sampling I will use another **model that can find and detect rare events**. 



In [ ]:
from sklearn.ensemble import IsolationForest 

pipeline9 = Pipe([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer,),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
     ("smote", smt),
    ("Isolation Forest", IsolationForest(n_estimators= 500, 
                                         contamination = 0.2,
                                         max_samples= 5))
])

In [ ]:
pipeline9.fit(x_train, y_train)

In [ ]:
y_pred_iso = pipeline9.predict(x_test) 
y_pred_iso_clean = np.where(y_pred_iso == -1, 1, 0)

In [ ]:
np.unique(y_pred_iso_clean)

In [ ]:
balanced_accuracy_score(y_test, y_pred_iso_clean) #Our model still does not have the best balanced accuracy at 52.0%

In [ ]:
precision_score(y_test, y_pred_iso_clean) # precision is also very low 7.03%

In [ ]:
recall_score(y_test, y_pred_iso_clean, average = "binary") #however, our recall is much higher therefore, we have caught more our our fraud cases 51.7%

In [ ]:
confusion_matrix_log = ConfusionMatrixDisplay.from_predictions( #creating confusion matrix diagram from predictions
    y_test,                                                     #comparing y test data to our predcited values from the logistic regression model
    y_pred_iso_clean
)

plt.title("Confusion matrix from Isolation tree")  #setting up the lablels 
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as imPipeline
from imblearn.over_sampling import SMOTE
from sklearn.neural_network import MLPClassifier

# Replace 'IsolationForest' with a supervised neural network
pipeline_nn = imPipeline([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
    ("smote", SMOTE(random_state=42)),  # Handles class imbalance
    ("nn", MLPClassifier(
        hidden_layer_sizes=(64,32),
        activation='relu',
        solver='adam',
        max_iter=200,
        random_state=42,
        early_stopping=True
    ))
])

In [ ]:
pipeline_nn.fit(x_train,y_train)

In [ ]:
y_pred_nn = pipeline_nn.predict(x_test)

In [ ]:
balanced_accuracy_score(y_test, y_pred_nn) #is lower than our logistic regression with classes weighed equally 

In [ ]:
precision_score(y_test, y_pred_nn) #also very low 0

In [ ]:
recall_score(y_test, y_pred_nn) #low like our logistic regression at 0